# Getting Started with Verily Workbench
_We recommend that all new users read this notebook to learn the basics of using Verily Workbench_

Below you will find recommended set up instructions for configuring your workspace and learning more about Verily Workbench before you begin your analysis.

This notebook was run using a standard JupyterLab Compute Engine VM with the default configuration and Python 3 kernel. If using other kernels, the default environment(env) variables might be different.

## Tutorial's Objectives

This tutorial is divided into the following sections

1. **CLI** How to use the Workbench CLI
2. **Workspace Bucket:** How to access your workspace bucket, create additional buckets and save files to your bucket
3. **WORKSPACE_CDR:** How to access AOU WORKSPACE_CDR.
4. **How to run** `/home/jupyter/workspace/aou-tutorial-notebooks/Setting_Env_Variables.ipynb and Setting_Env_Variables_p2.ipynb` to set up main env variables. **You must run this notebook to set environment variables, otherwise your notebooks will fail. Be review this tutorial entirely.**

## Using the Workbench CLI

First we will demonstrate how you can use the Workbench CLI. You can learn more about the Workbench CLI and the list of commands here: [Command-line interface overview](https://support.workbench.verily.com/docs/guides/cli/cli_intro/#what-can-you-do-with-the-cli-in-verily-workbench)

Using the Workbench CLI is optional, but is useful for managing Workbench resources. We will create a variable for the **GCP Project ID**, corresponding to your workspace. Your workspace bucket lives inside this GCP Project. Every Verily Workbench workspace maps to a GCP project.

We can use the workbench CLI to find our GCP Project ID, as shown below.

**If using Python 3 kernel, some env variables are already available**

In [3]:
%run /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Setting_Env_Variables.ipynb

Found bucket: id=rw-migration-aou-rw-f7a4d148, bucketName=rw-migration-aou-rw-f7a4d148
-> Assigned migration variables (ID: rw-migration-aou-rw-f7a4d148)
Found bucket: id=workspace-bucket, bucketName=workspace-bucket-wb-perky-cabbage-8342
✅ Successfully identified latest dataset: wb-silky-artichoke-2408.C2024Q3R9

Variables extracted:
GOOGLE_CLOUD_PROJECT: wb-perky-cabbage-8342
WORKSPACE_BUCKET: gs://workspace-bucket-wb-perky-cabbage-8342
WORKSPACE_TEMP_BUCKET: NOT FOUND
WORKSPACE_CDR: wb-silky-artichoke-2408.C2024Q3R9
bucket_aou_tutorial: NOT FOUND
bucket_id_aou_tutorial: NOT FOUND
bucket_migrated: gs://rw-migration-aou-rw-f7a4d148
bucket_id_migrated: rw-migration-aou-rw-f7a4d148

✅ Saved to /home/jupyter/.bashrc
C2024Q3R9 BQ_DATASET
Multi-trait-GWAS-in-admixed-populations GIT_REPO
prep_C2024Q3R9 BQ_DATASET
rw-migration-aou-rw-f7a4d148 GCS_BUCKET
workspace-bucket GCS_BUCKET
GOOGLE_CLOUD_PROJECT = wb-perky-cabbage-8342
WORKSPACE_BUCKET = gs://workspace-bucket-wb-perky-cabbage-8342
WORK

In [4]:
%run /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Setting_Env_Variables_p2.ipynb

WORKSPACE_CDR = wb-silky-artichoke-2408.C2024Q3R9
WORKSPACE_BUCKET = gs://workspace-bucket-wb-perky-cabbage-8342
GOOGLE_PROJECT = wb-perky-cabbage-8342
Done! 10 variables saved to: /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Setting_Env_Variables_p2.R
Done! 10 variables saved to: /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Setting_Env_Variables.sas


In [5]:
!echo $GOOGLE_CLOUD_PROJECT

wb-perky-cabbage-8342


In [6]:
import os
os.environ["GOOGLE_CLOUD_PROJECT"]

'wb-perky-cabbage-8342'

In [7]:
!echo $GOOGLE_PROJECT

wb-perky-cabbage-8342


In [8]:
!echo $WORKBENCH_USER_EMAIL

remi.rogulski@researchallofus.org


**If using other kernels, you may have to manually set up or extract some variables as shown below**

In [9]:
!wb workspace describe --format=json | jq -r .'googleProjectId'

wb-perky-cabbage-8342


We can set an environment variable for our GCP Project ID. Copy the GCP Project ID below.

In [ ]:
# GOOGLE_CLOUD_PROJECT = !wb workspace describe --format=json | jq -r .'googleProjectId'
# GOOGLE_CLOUD_PROJECT

This is a common approach to set a variable in our environment for easy reference in a notebook.

However for our purposes, we have already generated variables and will import them using the **Setting_Env_variables.ipynb & Setting_Env_variables_p2.ipynb** which will be sourced at the top of every notebook you run and is demonstarted at the end of this notebook.

## Accesing your workspace bucket

Now we will walk through how you can create a workspace bucket and save files to a bucket. You can run the following cell to confirm that you are using the workspace that you intend to work in. Uncomment the cell and run to confirm. (You can also run `wb workspace list` to see your list of workspaces).

In [11]:
# !wb status

You can view all of the resources in your workspace with the command `!wb resource list`. We recommend you take a look at your workspace resources prior to creating default resources. You will see that a workspace bucket has been created for you automatically named `workspace_bucket`.

In [12]:
!wb resource list

ID                                       RESOURCE TYPE  STEWARDSHIP TYPE  DESCRIPTION                             
C2024Q3R9                                BQ_DATASET     REFERENCED        (unset)                                 
Multi-trait-GWAS-in-admixed-populations  GIT_REPO       REFERENCED        Paper Project                           
prep_C2024Q3R9                           BQ_DATASET     REFERENCED        (unset)                                 
rw-migration-aou-rw-f7a4d148             GCS_BUCKET     CONTROLLED        RW migration bucket for workspace aou...
workspace-bucket                         GCS_BUCKET     CONTROLLED        Primary workspace bucket for storing ...


As you can see, this workspace already has a workspace bucket titled **aou-tutorial-notebooks**. All of the tutorial notebook files are saved to this bucket.

You can optionally create additional buckets, which the instructions below detail.

## Create Cloud Storage buckets

The instructions below indicate how to create two Cloud Storage buckets in your workspace with the following workspace reference names:

* `workspace-bucket`: This is your primary workspace bucket. You can use your workspace bucket to share notebooks with other workbench users and store files. We will show you how to save files to your workspace bucket below.
* `temporary-workspace-bucket`: This is a temporary workspace bucket that will auto-delete after 2 weeks. We recommend you write output files to this bucket if you do not intend to keep them longer than 2 weeks. Any file in this bucket will be automatically deleted two weeks after it is written. This alleviates the need for you to remember to clean up temporary and example files manually. For example, if you have a job with intermediate outputs you don't plan to reference again, you can save the intermediate files in your `temporary-workspace-bucket`

In [13]:
!wb resource resolve --name workspace-bucket || wb resource create gcs-bucket \
    --name=workspace-bucket \
    --cloning=COPY_NOTHING \
    --description="Primary workspace bucket for storing files"

gs://workspace-bucket-wb-perky-cabbage-8342


In [14]:
!wb resource resolve --name temporary-workspace-bucket || wb resource create gcs-bucket \
    --name=temporary-workspace-bucket \
    --cloning=COPY_NOTHING \
    --auto-delete=14 \
    --description="Bucket for temporary storage of file data. Send outputs here for automatic cleanup after two weeks."

Resource not found: temporary-workspace-bucket
Successfully added controlled GCS bucket.
ID:           temporary-workspace-bucket
Description:  Bucket for temporary storage of file data. Send outputs here for automatic cleanup after two weeks.
Type:         GCS_BUCKET
Stewardship:  CONTROLLED
Cloning:      COPY_NOTHING
Access scope: SHARED_ACCESS
Managed by:   USER
Region:       US-CENTRAL1
Properties:   class Properties {
    []
}
GCS bucket name: temporary-workspace-bucket-wb-perky-cabbage-8342
Location:        US-CENTRAL1
# Objects:       0


**Note**

You can use the UI to create the resources like buckets or bigquery datasets. One difference you might have to notice is that the bucket created via the UI will have "Cloning: COPY_RESOURCE" feature, meaning the bucket will be duplicated as well when the workspace is being duplicated. In contrast,using vwb CLI, you can control this feature as shown in the examples above: both "workspace-bucket" and "temporary-workspace-bucket" have "Cloning: COPY_NOTHING", meaning they will not be duplicated.

Next, we will save our workspace bucket path as a variable. You can resolve the path to your workspace bucket by running the following command, which gives you the abililty to easily reference your bucket. Learn more about referenced resources on the [Verily Support Hub](https://support.workbench.verily.com/docs/guides/research_data/resource_intro/) and on the [Verily Workbench GitHub repository](https://github.com/verily-src/workbench-examples)

### workspace-bucket

In [15]:
import os

In [16]:
WORKSPACE_BUCKET_URL = !wb resource resolve --id workspace-bucket
WORKSPACE_BUCKET_URL = WORKSPACE_BUCKET_URL[0]
WORKSPACE_BUCKET_URL

'gs://workspace-bucket-wb-perky-cabbage-8342'

In [17]:
%env WORKSPACE_BUCKET = $WORKSPACE_BUCKET_URL

env: WORKSPACE_BUCKET=gs://workspace-bucket-wb-perky-cabbage-8342


In [18]:
print(os.environ.get("WORKSPACE_BUCKET"))

gs://workspace-bucket-wb-perky-cabbage-8342


In [19]:
!wb resource describe --id workspace-bucket --format JSON

{
  "uuid" : "e781d107-6b34-4fd9-84c8-f8b51e6d86d6",
  "id" : "workspace-bucket",
  "folderId" : null,
  "description" : "Primary workspace bucket for storing files",
  "resourceType" : "GCS_BUCKET",
  "stewardshipType" : "CONTROLLED",
  "cloningInstructions" : "COPY_NOTHING",
  "accessScope" : "SHARED_ACCESS",
  "managedBy" : "USER",
  "region" : "US-CENTRAL1",
  "privateUserName" : null,
  "privateUserRole" : null,
  "properties" : [ ],
  "createdBy" : "remi.rogulski@researchallofus.org",
  "bucketName" : "workspace-bucket-wb-perky-cabbage-8342",
  "location" : "US-CENTRAL1",
  "numObjects" : 0
}


### temporary-workspace-bucket

In [20]:
TEMP_WORKSPACE_BUCKET_URL = !wb resource resolve --id temporary-workspace-bucket
TEMP_WORKSPACE_BUCKET_URL = TEMP_WORKSPACE_BUCKET_URL[0]
TEMP_WORKSPACE_BUCKET_URL

'gs://temporary-workspace-bucket-wb-perky-cabbage-8342'

In [21]:
%env TEMP_WORKSPACE_BUCKET = $TEMP_WORKSPACE_BUCKET_URL

env: TEMP_WORKSPACE_BUCKET=gs://temporary-workspace-bucket-wb-perky-cabbage-8342


In [22]:
print(os.environ.get("TEMP_WORKSPACE_BUCKET"))

gs://temporary-workspace-bucket-wb-perky-cabbage-8342


### Verification

We can run the cell below to check the buckets

In [23]:
import json
import subprocess

# Get all resources
resources_json = subprocess.run(
    ["wb", "resource", "list", "--format=json"],
    capture_output=True,
    text=True,
    check=True
).stdout
resources = json.loads(resources_json)

# Extract ALL GCS_BUCKET resources
all_buckets = [r for r in resources if r["resourceType"] == "GCS_BUCKET"]

print(f"Found {len(all_buckets)} GCS buckets:")
for i, bucket in enumerate(all_buckets, 1):
    print(f"{i}. ID: {bucket['id']}, Name: {bucket['bucketName']}")
    
# Or as gs:// URLs
gs_urls = [f"gs://{b['bucketName']}" for b in all_buckets]
print(f"\nGS URLs: {gs_urls}")

Found 3 GCS buckets:
1. ID: rw-migration-aou-rw-f7a4d148, Name: rw-migration-aou-rw-f7a4d148
2. ID: temporary-workspace-bucket, Name: temporary-workspace-bucket-wb-perky-cabbage-8342
3. ID: workspace-bucket, Name: workspace-bucket-wb-perky-cabbage-8342

GS URLs: ['gs://rw-migration-aou-rw-f7a4d148', 'gs://temporary-workspace-bucket-wb-perky-cabbage-8342', 'gs://workspace-bucket-wb-perky-cabbage-8342']


## Check AOU bigquery dataset

Check what bigquery datasets are available already.

In [25]:
!wb resource list

ID                                       RESOURCE TYPE  STEWARDSHIP TYPE  DESCRIPTION                             
C2024Q3R9                                BQ_DATASET     REFERENCED        (unset)                                 
Multi-trait-GWAS-in-admixed-populations  GIT_REPO       REFERENCED        Paper Project                           
prep_C2024Q3R9                           BQ_DATASET     REFERENCED        (unset)                                 
rw-migration-aou-rw-f7a4d148             GCS_BUCKET     CONTROLLED        RW migration bucket for workspace aou...
temporary-workspace-bucket               GCS_BUCKET     CONTROLLED        Bucket for temporary storage of file ...
workspace-bucket                         GCS_BUCKET     CONTROLLED        Primary workspace bucket for storing ...


In [26]:
dataset_id='C2024Q3R9'

In [27]:
project_id = os.environ.get('GOOGLE_CLOUD_PROJECT')
project_id

'wb-perky-cabbage-8342'

In [28]:
WORKSPACE_CDR = f"{project_id}.{dataset_id}"
WORKSPACE_CDR

'wb-perky-cabbage-8342.C2024Q3R9'

In [29]:
# Set environment variable
os.environ['WORKSPACE_CDR'] = WORKSPACE_CDR

In [30]:
!echo $WORKSPACE_CDR

wb-perky-cabbage-8342.C2024Q3R9


Or we can run the cell below for this purpose.

In [31]:
import re
import json
import subprocess
import os

resources_json = subprocess.run(
    ["wb", "resource", "list", "--format=json"],
    capture_output=True,
    text=True,
    check=True
).stdout

resources = json.loads(resources_json)

for r in resources:
    if r["resourceType"] in ["BQ_DATASET", "BIGQUERY_DATASET"]:
        dataset = r["datasetId"]

        if re.match(r"^C\d{4}Q\d+R\d+$", dataset):
            os.environ["WORKSPACE_CDR"] = f"{r['projectId']}.{dataset}"
            break

print(os.environ.get("WORKSPACE_CDR"))

wb-silky-artichoke-2408.C2024Q3R9


## Create bigquery datasets

Again, we can use the UI to create a bigquery dataset or can run the cell below to create one

In [32]:
!wb resource resolve --name dataset_test2 || wb resource create bq-dataset \
    --name=dataset_test2 \
    --cloning COPY_REFERENCE \
    --description="bigquery dataset test COPY_REFERENCE"

Resource not found: dataset_test2
Successfully added controlled BigQuery dataset.
ID:           dataset_test2
Description:  bigquery dataset test COPY_REFERENCE
Type:         BQ_DATASET
Stewardship:  CONTROLLED
Cloning:      COPY_REFERENCE
Access scope: SHARED_ACCESS
Managed by:   USER
Region:       us-central1
Properties:   class Properties {
    []
}
GCP project id:      wb-perky-cabbage-8342
BigQuery dataset id: dataset_test2
Location:            us-central1
# Tables:            0


**Check the description of this new dataset**

In [33]:
!wb resource describe --id dataset_test2 --format JSON

{
  "uuid" : "ec2ca892-ba0a-40b6-9735-44f4ae73057b",
  "id" : "dataset_test2",
  "folderId" : null,
  "description" : "bigquery dataset test COPY_REFERENCE",
  "resourceType" : "BQ_DATASET",
  "stewardshipType" : "CONTROLLED",
  "cloningInstructions" : "COPY_REFERENCE",
  "accessScope" : "SHARED_ACCESS",
  "managedBy" : "USER",
  "region" : "us-central1",
  "privateUserName" : null,
  "privateUserRole" : null,
  "properties" : [ ],
  "createdBy" : "remi.rogulski@researchallofus.org",
  "projectId" : "wb-perky-cabbage-8342",
  "datasetId" : "dataset_test2",
  "location" : "us-central1",
  "numTables" : 0
}


**Note**

This dataset has `cloningInstructions" : "COPY_REFERENCE"` feature, meaning this dataset will be only referenced but not copied when the workspace is beding duplicated.

## How to automatically set up some env variables including

* GOOGLE_CLOUD_PROJECT
* WORKSPACE_BUCKET
* WORKSPACE_TEMP_BUCKET
* WORKSPACE_CDR

In a new notebook, first run the cell below and only need to run it once. We can open it to see the details.

In [35]:
%run /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Setting_Env_Variables.ipynb

Found bucket: id=rw-migration-aou-rw-f7a4d148, bucketName=rw-migration-aou-rw-f7a4d148
-> Assigned migration variables (ID: rw-migration-aou-rw-f7a4d148)
Found bucket: id=temporary-workspace-bucket, bucketName=temporary-workspace-bucket-wb-perky-cabbage-8342
Found bucket: id=workspace-bucket, bucketName=workspace-bucket-wb-perky-cabbage-8342
✅ Successfully identified latest dataset: wb-silky-artichoke-2408.C2024Q3R9

Variables extracted:
GOOGLE_CLOUD_PROJECT: wb-perky-cabbage-8342
WORKSPACE_BUCKET: gs://workspace-bucket-wb-perky-cabbage-8342
WORKSPACE_TEMP_BUCKET: gs://temporary-workspace-bucket-wb-perky-cabbage-8342
WORKSPACE_CDR: wb-silky-artichoke-2408.C2024Q3R9
bucket_aou_tutorial: NOT FOUND
bucket_id_aou_tutorial: NOT FOUND
bucket_migrated: gs://rw-migration-aou-rw-f7a4d148
bucket_id_migrated: rw-migration-aou-rw-f7a4d148

✅ Saved to /home/jupyter/.bashrc
C2024Q3R9 BQ_DATASET
Multi-trait-GWAS-in-admixed-populations GIT_REPO
dataset_test2 BQ_DATASET
prep_C2024Q3R9 BQ_DATASET
rw-mig

In [ ]:
%run /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Setting_Env_Variables_p2.ipynb

### Disk Storage

In addition to your workspace bucket, you also have a storage disk that is used for local file storage. When you pause your application, your disk persists and you are responsible for disk cloud costs. When you delete your application, your disk is deleted. You can learn more about disks on the [Verily Support Hub](https://support.workbench.verily.com/docs/guides/cloud_apps/compute_config_options/#disk-storage)

## Next Steps

1. Review `access-aou-data.ipynb`: This notebook is your **primary guide** for accessing and working with the All of Us research data.
2. Review the other tutorials available in this workspace, including **`How to Get Started with Registered Tier Data (v8)`** and **`Beginner Intro to AoU Data and the Workbench`**.

## Provenance

Below are optional provenance steps we recommend including in your notebooks to generate information about your notebook environment:

In [36]:
!date

Thu Jul  2 10:11:15 PM UTC 2026


In [37]:
#Conda and pip installed packages:
!conda env export

name: base
channels:
  - conda-forge
dependencies:
  - _libgcc_mutex=0.1=conda_forge
  - _openmp_mutex=4.5=2_gnu
  - archspec=0.2.5=pyhd8ed1ab_0
  - backports.zstd=1.3.0=py313h18e8e13_0
  - boltons=25.0.0=pyhd8ed1ab_0
  - brotli-python=1.2.0=py313hf159716_1
  - bzip2=1.0.8=hda65f42_8
  - c-ares=1.34.6=hb03c661_0
  - ca-certificates=2026.1.4=hbd8a1cb_0
  - certifi=2026.1.4=pyhd8ed1ab_0
  - cffi=2.0.0=py313hf46b229_1
  - charset-normalizer=3.4.4=pyhd8ed1ab_0
  - conda=26.1.0=py313h78bf25f_0
  - conda-libmamba-solver=25.11.0=pyhd8ed1ab_1
  - conda-package-handling=2.4.0=pyh7900ff3_2
  - conda-package-streaming=0.12.0=pyhd8ed1ab_0
  - cpp-expected=1.3.1=h171cf75_0
  - distro=1.9.0=pyhd8ed1ab_1
  - fmt=12.1.0=hff5e90c_0
  - frozendict=2.4.7=py313h07c4f96_0
  - h2=4.3.0=pyhcf101f3_0
  - hpack=4.1.0=pyhd8ed1ab_0
  - hyperframe=6.1.0=pyhd8ed1ab_0
  - icu=78.2=h33c6efd_0
  - idna=3.11=pyhd8ed1ab_0
  - jsonpatch=1.33=pyhd8ed1ab_1
  - jsonpointer=3.0.0=pyhcf101f3_3
  - keyutils=1.6.3=hb9d3cd8_0
 

In [38]:
#JupyterLab extensions:
!jupyter labextension list

JupyterLab v4.5.6
/opt/conda/envs/jupyter/share/jupyter/labextensions
        jupyterlab_pygments v0.3.0 enabled OK (python, jupyterlab_pygments)
        jupyterlab-jupytext v1.4.6 enabled OK (python, jupytext)
        jupyterlab-plotly v5.24.1 enabled  X
        nbdime-jupyterlab v3.0.4 enabled OK
        aou-jupyterlab v0.1.0 enabled OK (python, aou_jupyterlab)
        jupyterlab-snippets v0.4.1 enabled OK (python, jupyterlab-snippets)
        server-proxy-notif v0.1.0 enabled OK (python, server_proxy_notif)
        @jupyterhub/jupyter-server-proxy v4.5.0 enabled OK
        @jupyter-widgets/jupyterlab-manager v5.0.15 enabled OK (python, jupyterlab_widgets)
        @jupyter-notebook/lab-extension v7.5.5 enabled OK
        @jupyterlab/git v0.52.0 enabled OK (python, jupyterlab-git)


   The following extensions may be outdated or specify dependencies that are incompatible with the current version of jupyterlab:
        jupyterlab-plotly
        
   If you are a user, check if an update

In [39]:
#Number of cores:
!grep ^processor /proc/cpuinfo | wc -l

4


In [40]:
#Memory:
!grep "^MemTotal:" /proc/meminfo

MemTotal:       26661080 kB
